# Step 5.h — Climate and soil suitability on ERA5, global (EC-Earth3 grid): final rule and validation

**Final binary suitability = climate suitable AND soil suitable**, per crop and GAEZ input level (low, high), each
condition on its own scale (`cu.suitable`):
- **climate**: `suit_bin` of 4.g on ERA5 1981–2014 = SVM probability ≥ `cu.CLIMATE_P` = 0.5 AND growing-period
  temperature ≥ the FAO ECOCROP TMIN of the crop (`cu.climate_suitable`). The SVM probability is brought to equal
  class priors in 4.g, so 0.5 is the decision threshold of the classifier: above it the climate of the cell is closer
  to the climates where the crop is grown than to those of the cropland where it is not. Not suitable where the
  climate is novel (MESS < 0, NT2 > 1 or no support vector within the reach of the kernel);
- **soil**: soil suitability rating of 2.g (HWSD v2.01 × GAEZ v5 Module IV, rainfed, % of the attainable yield, mean
  over the cell, permafrost = 0) ≥ `cu.SOIL_MIN` = 25, the GAEZ classes "moderate" (25–40) or better; "marginal"
  (10–25), "very marginal" (0–10) and "not suitable" are left out. Land with no soil rating (ice, water) = 0.

Why not the product climate × soil (the previous version): GAEZ multiplies an agro-climatic index that is a fraction
of the attainable yield by a soil rating that is also a fraction of it; the SVM probability is a classification
score, so the product has no physical meaning and its trade-off between climate and soil is arbitrary. Thresholds
calibrated on 95 % of the MIRCA2000 area were also tested: on the climate probability alone they fall where it
saturates (potato 0.973), and the hosing effect then depends on the threshold (+40 to +65 % for potato). With
CLIMATE_P = 0.5 and SOIL_MIN = 25 the hosing effects are stable between 0.4 and 0.6 and between soil 10 and 25 (tests
of 5 November 2026, method note).

**Checks**:
1. **MIRCA2000**: AUC of the climate probability and of the soil rating on the training cells of 3.g; share of the
   MIRCA2000 harvested area and of the presence cells in suitable cells, and what the rest is lost to (soil below
   SOIL_MIN, or climate below CLIMATE_P); TSS, sensitivity, specificity of the binary map on the training cells;
2. **GAEZ v5** (SIX, HP8100 = 1981–2000, AgERA5, rainfed, high input HRLM / low input LRLM; wheat not split into
   winter / spring; rice = wetland rice RCW): class mid-points averaged over the cell, suitable = mean SI ≥ 0.25 (the
   same class limit as the soil); agreement of the binary maps; correlation with the climate probability;
3. **too cold for the crop** (3.h): suitable cells below the FAO ECOCROP absolute minimum temperature, by region.

Outputs in `<out_root>/5_climate_soil/<route>/<variant>_<radiation>/`: `climate_soil_<crop>.nc`,
`validation_<set>.csv`, `areas_<set>.csv`, `gaez_comparison_<set>.csv`, `rule_<set>.csv`, maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr, rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the set of 4.g
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")   # as 4.g (rad_era5 once the rsds of the hosing runs arrives);
                                                     # set for one run with the environment (CHOSEN_RAD=rad_era5)
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
LEVELS = ["low", "high"]           # GAEZ input level of the soil rating (2.g) and of GAEZ v5
GAEZ_CODE = {"Maize_rf": "MZE", "Potato_rf": "WPO", "Soybean_rf": "SOY", "Rice_rf": "RCW",
             "Wheat_rf": "WHE", "Wheat-winter_rf": "WHE", "Wheat-spring_rf": "WHE"}
GAEZ_WSIM = {"high": "HRLM", "low": "LRLM"}   # rainfed, high / low input
GAEZ_MIN = cu.SOIL_MIN / 100       # GAEZ v5 suitable = mean SI of the cell >= 0.25 (the class limit of the soil)
ECOCROP = cu.ECOCROP_TMIN          # FAO ECOCROP absolute minimum temperature, as 3.h and 4.g

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
CLIM_DIR = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
GAEZ_DIR = cu.data_dir(CFG, "gaez_v5_hist")
OUT = cu.out_dir(CFG, "5_climate_soil", ROUTE, SET)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
CRO_ANY = cu.mirca_cropland(CFG).values                 # any MIRCA2000 cropland (no threshold), as 3.h
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG))
LAND_MHA = LAND_HA * LAND / 1e6
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
REGIONS = {"MIRCA cropland": LAND & CRO_ANY,
           "outside cropland": LAND & ~CRO_ANY,
           "north of 50 N, outside cropland": LAND & ~CRO_ANY & (LAT2 > 50),
           "Tibetan plateau, outside cropland": LAND & ~CRO_ANY & (LAT2 > 27) & (LAT2 < 40) & (LON2 > 75) & (LON2 < 105),
           "Antarctica": LAND & (LAT2 < -60)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
GAEZ_FILE = {(lv, c): GAEZ_DIR / f"GAEZ-V5.RES05-SIX.HP8100.AGERA5.HIST.{GAEZ_CODE[c]}.{GAEZ_WSIM[lv]}.tif"
             for lv in LEVELS for c in CROPS}
missing = sorted({p.name for p in GAEZ_FILE.values() if not p.exists()})
print(f"set {SET} | rule: climate p >= {cu.CLIMATE_P} and >= ECOCROP TMIN, soil >= {cu.SOIL_MIN} | climate {CLIM_DIR} | outputs {OUT}\n"
      f"GAEZ v5 files missing (comparison skipped): {missing or 'none'}")

## 1. Final suitability, and the layers on the training cells

In [ ]:
SI_MID = np.array([np.nan, 0.925, 0.775, 0.625, 0.475, 0.325, 0.175, 0.05, 0.0, np.nan])   # GAEZ SIX class -> SI mid-point
_IDX = {}                                                                                 # (0 no data, 8 not suitable, 9 water)


def gaez_on_grid(path):
    # GAEZ v5 SIX (5 arc-min) -> mean SI mid-point of the pixels of each EC-Earth3 cell, cos-latitude weights
    with rasterio.open(path) as src:
        a, T = src.read(1), src.transform
    key = (a.shape, tuple(T))
    if key not in _IDX:
        lat = T.f + T.e * (np.arange(a.shape[0]) + 0.5); lon = T.c + T.a * (np.arange(a.shape[1]) + 0.5)
        I, _ = cu.cell_of(lat, np.zeros_like(lat), GRID); _, J = cu.cell_of(np.zeros_like(lon), lon, GRID)
        _IDX[key] = ((I[:, None] * GRID["lon"].size + J[None, :]).ravel(), np.repeat(np.cos(np.radians(lat)), a.shape[1]))
    flat, w = _IDX[key]
    s = SI_MID[np.minimum(a, 9)].ravel()
    ok = np.isfinite(s)
    num = np.bincount(flat[ok], weights=s[ok] * w[ok], minlength=LAND.size)
    den = np.bincount(flat[ok], weights=w[ok], minlength=LAND.size)
    with np.errstate(invalid="ignore", divide="ignore"):
        g = np.where(den > 0, num / den, np.nan).reshape(LAND.shape)
    return np.where(LAND, g, np.nan)


mha = lambda mask: float(np.nansum(LAND_MHA * mask))
val_rows, area_rows, gaez_rows, MAPS = [], [], [], {}
for crop in CROPS:
    with xr.open_dataset(CLIM_DIR / f"suitability_{crop}.nc") as ds:
        clim = ds["suit_proba"].sel(scenario="ERA5").values.astype(float)
        clim_ok = ds["suit_bin"].sel(scenario="ERA5").values == 1             # probability >= 0.5 and >= ECOCROP TMIN
        novel = ds["novel"].sel(scenario="ERA5").values == 1
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv", usecols=["lat", "lon", "label"])
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    y = tr.label.values
    tmin, which = ECOCROP[crop]
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        T = cu.growing_temperature(ds, which)
    cold = LAND & np.isfinite(T) & (T < tmin)
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    pres = LAND & (crop_ha > CELL_AREA_THR * LAND_HA)          # presences of 3.g

    layers = {"climate": clim}
    soil, suit, gaez = {}, {}, {}
    for lv in LEVELS:
        soil[lv] = np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), np.nan)      # 0-100, no rating -> 0
        layers[f"soil ({lv})"] = soil[lv]
        suit[lv] = LAND & cu.suitable(clim_ok, soil[lv])
        if GAEZ_FILE[(lv, crop)].exists():
            gaez[lv] = gaez_on_grid(GAEZ_FILE[(lv, crop)])
            layers[f"GAEZ v5 ({lv})"] = gaez[lv]
    for name, a in layers.items():
        v = a[ii, jj]; ok = np.isfinite(v)
        val_rows.append({"crop": crop, "layer": name, "training cells": int(ok.sum()), "AUC": roc_auc_score(y[ok], v[ok]),
                         "mean on presences": v[ok][y[ok] == 1].mean(), "mean on absences": v[ok][y[ok] == 0].mean()})
    for lv in LEVELS:
        s = suit[lv]
        sens, spec = s[ii, jj][y == 1].mean(), (~s[ii, jj])[y == 0].mean()
        soil_ok = np.nan_to_num(soil[lv]) >= cu.SOIL_MIN
        r = {"crop": crop, "level": lv, "suitable land (Mha)": mha(s),
             "MIRCA area of the crop in suitable cells (%)": 100 * crop_ha[s].sum() / crop_ha.sum(),
             "presence cells suitable (%)": 100 * (s & pres).sum() / pres.sum(),
             "crop area lost to the soil (%)": 100 * crop_ha[LAND & ~soil_ok].sum() / crop_ha.sum(),
             "crop area lost to the climate only (%)": 100 * crop_ha[LAND & soil_ok & ~s].sum() / crop_ha.sum(),
             "of which to the ECOCROP TMIN (%)": 100 * crop_ha[LAND & soil_ok & ~s & (clim >= cu.CLIMATE_P) & cold].sum() / crop_ha.sum(),
             "TSS (training cells)": sens + spec - 1, "sensitivity": sens, "specificity": spec,
             "novel climate (Mha)": mha(LAND & novel)}
        for reg, rm in REGIONS.items():
            r[f"{reg} (Mha)"] = mha(s & rm)
        r["below TMIN, outside cropland (Mha)"] = mha(s & cold & REGIONS["outside cropland"])
        r["below TMIN, north of 50 N outside cropland (Mha)"] = mha(s & cold & REGIONS["north of 50 N, outside cropland"])
        area_rows.append(r)
        if lv in gaez:
            g = gaez[lv]
            both = np.isfinite(g) & np.isfinite(clim)
            sg = both & (g >= GAEZ_MIN)
            gaez_rows.append({"crop": crop, "level": lv, "cells": int(both.sum()),
                              "Pearson r climate p vs GAEZ SI": np.corrcoef(clim[both], g[both])[0, 1],
                              "Spearman rho climate p vs GAEZ SI": spearmanr(clim[both], g[both])[0],
                              "both suitable (Mha)": mha(sg & s), "ours only (Mha)": mha(both & s & ~sg),
                              "GAEZ only (Mha)": mha(sg & ~s),
                              "GAEZ: MIRCA area in suitable cells (%)": 100 * crop_ha[sg].sum() / crop_ha.sum()})
    MAPS[crop] = {"clim": clim, "soil": soil, "suit": suit, "cold": cold, "novel": novel, "gaez": gaez}
    dv = {"climate": (("lat", "lon"), clim.astype("float32"), {"long_name": "climate suitability probability (4.g, ERA5, equal priors), 0 where novel"}),
          "novel": (("lat", "lon"), novel.astype("int8"), {"long_name": "novel climate (MESS < 0 or NT2 > 1 or Kmax < 0.01)"}),
          "below_tmin": (("lat", "lon"), cold.astype("int8"), {"long_name": f"growing-period temperature ({which}) below the ECOCROP TMIN {tmin} degC"})}
    for lv in LEVELS:
        dv[f"soil_{lv}"] = (("lat", "lon"), soil[lv].astype("float32"), {"long_name": f"soil suitability rating (2.g, 0-100), {lv} input"})
        dv[f"suitable_{lv}"] = (("lat", "lon"), np.where(LAND, suit[lv], np.nan).astype("float32"),
                                {"long_name": f"final suitability, {lv} input: climate suitable (p >= {cu.CLIMATE_P}, >= ECOCROP TMIN) and soil >= {cu.SOIL_MIN}"})
        if lv in gaez:
            dv[f"gaez_v5_{lv}"] = (("lat", "lon"), gaez[lv].astype("float32"),
                                   {"long_name": f"GAEZ v5 SIX HP8100 {GAEZ_CODE[crop]} {GAEZ_WSIM[lv]}, mean class mid-point"})
    xr.Dataset(dv, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "set": SET, "route": ROUTE, "climate": f"{ERA5_NAME} {ERA5_PERIOD[0]}-{ERA5_PERIOD[1]}",
                      "climate_p": cu.CLIMATE_P, "soil_min": cu.SOIL_MIN}).to_netcdf(OUT / f"climate_soil_{crop}.nc")
    print(f"{crop}: suitable " + ", ".join(f"{lv} {mha(suit[lv]):.0f} Mha" for lv in LEVELS), flush=True)

## 2. Results

(1) AUC on the training cells: climate, soil, GAEZ v5. (2) The final binary map: areas, MIRCA2000 area and presence
cells kept and what the rest is lost to, TSS on the training cells, cold cells by region. (3) Agreement with GAEZ v5.

In [ ]:
val = pd.DataFrame(val_rows).set_index(["crop", "layer"])
areas = pd.DataFrame(area_rows).set_index(["crop", "level"])
gaez = pd.DataFrame(gaez_rows).set_index(["crop", "level"]) if gaez_rows else pd.DataFrame()
val.to_csv(OUT / f"validation_{SET}.csv"); areas.to_csv(OUT / f"areas_{SET}.csv"); gaez.to_csv(OUT / f"gaez_comparison_{SET}.csv")
pd.DataFrame({"climate_p": [cu.CLIMATE_P], "soil_min": [cu.SOIL_MIN]}).to_csv(OUT / f"rule_{SET}.csv", index=False)
with pd.option_context("display.max_rows", 300, "display.max_columns", 30, "display.width", 250):
    print("(1) AUC on the training cells (MIRCA2000 presences vs cropland absences, ERA5)")
    display(val["AUC"].unstack("layer").round(3))
    print(f"(2) final suitability (climate p >= {cu.CLIMATE_P} and soil >= {cu.SOIL_MIN})")
    display(areas.round(2))
    print("(3) final map vs GAEZ v5 (GAEZ suitable = mean SI >= 0.25)")
    display(gaez.round(3))

## 3. Maps

In [ ]:
cls_cmap = ListedColormap(["#e6e6e6", "#4daf4a", "#e41a1c", "#7f7f7f"])   # not suitable | suitable | suitable but below TMIN | novel
for crop in CROPS:
    M = MAPS[crop]
    top = [("climate probability (ERA5)", M["clim"], (0, 1), "YlGn")] + \
          [(f"soil rating ({lv})", M["soil"][lv], (0, 100), "YlOrBr") for lv in LEVELS] + \
          [(f"GAEZ v5 SI ({lv})", M["gaez"][lv], (0, 1), "YlGn") for lv in LEVELS[::-1] if lv in M["gaez"]][:1]
    fig, axes = plt.subplots(2, 4, figsize=(26, 7))
    for ax, (t, a, lim, cm) in zip(axes[0], top):
        im = ax.pcolormesh(GRID["lon"], GRID["lat"], a, vmin=lim[0], vmax=lim[1], cmap=cm, shading="nearest")
        ax.set_title(f"{crop}: {t}", fontsize=9); plt.colorbar(im, ax=ax, shrink=0.8)
    for ax, lv in zip(axes[1], LEVELS):
        s = M["suit"][lv]
        c = np.where(LAND, np.where(M["novel"], 3, np.where(s & M["cold"], 2, s.astype(float))), np.nan)
        ax.pcolormesh(GRID["lon"], GRID["lat"], c, vmin=-0.5, vmax=3.5, cmap=cls_cmap, shading="nearest")
        ax.set_title(f"{crop}, {lv} input: climate >= {cu.CLIMATE_P} and soil >= {cu.SOIL_MIN}; red: below TMIN, grey: novel", fontsize=9)
    for ax in axes[1][len(LEVELS):]:
        ax.axis("off")
    plt.savefig(OUT / f"climate_soil_maps_{SET}_{crop}.png", dpi=80, bbox_inches="tight"); plt.show()